# Particle tracking on an unstructured grid

This example demonstrates forward tracking with MF6 PRT on a Voronoi grid. Two particles are tracked through the domain and their temperatures interpolated along the path. This scenario provides a sneak peak of one of tomorrow morning's topics, heat transport. The scenario is adapted from [this MF6 example problem](https://modflow6-examples.readthedocs.io/en/master/_notebooks/ex-gwe-prt.html).

First, define units.

In [ ]:
time_unit = "years"
length_unit = "meters"

Load a Voronoi grid from a pre-existing binary grid file. The grid covers a 2000 × 1000m rectangular domain.

In [ ]:
from pathlib import Path

from flopy.mf6.utils import MfGrdFile

grb = MfGrdFile(Path("../data/prt_voronoi/ex-gwe-prt-gwf.disv.grb"))
mg = grb.modelgrid

This model has CHD boundary conditions on the left, right, and bottom, and 3 wells: 2 pumping and 1 injection. We will use FloPy's grid intersection utilities to identify boundary cells.

In [ ]:
from flopy.utils import GridIntersect
from shapely.geometry import LineString, Point

xmin, xmax = 0.0, 2000.0
ymin, ymax = 0.0, 1000.0

gi = GridIntersect(mg)

cells_left = gi.intersect(
    LineString([(xmin, ymin), (xmin, ymax)]), geo_dataframe=False
)["cellids"]
cells_right = gi.intersect(
    LineString([(xmax, ymin), (xmax, ymax)]), geo_dataframe=False
)["cellids"]
cells_bottom = gi.intersect(
    LineString([(xmin, ymin), (xmax, ymin)]), geo_dataframe=False
)["cellids"]
cells_wel = [
    mg.intersect(p.x, p.y)
    for p in [Point(1200, 500), Point(700, 200), Point(1600, 700)]
]

Set up a base workspace for the scenario, and a nested workspace for the flow simulation.

In [ ]:
example_name = "prt_voronoi"
base_ws = Path("models") / example_name
gwf_name = f"{example_name}-gwf"
gwf_ws = base_ws / "gwf"
gwf_ws.mkdir(exist_ok=True, parents=True)

Now we can construct the flow simulation. We will speed through this without much commentary.

In [ ]:
import flopy

gwf_sim = flopy.mf6.MFSimulation(
    sim_name=gwf_name, version="mf6", exe_name="mf6", sim_ws=gwf_ws
)
gwf_tdis = flopy.mf6.ModflowTdis(
    gwf_sim, time_units=time_unit, perioddata=[[1.0, 1, 1.0]]
)
gwf = flopy.mf6.ModflowGwf(gwf_sim, modelname=gwf_name, save_flows=True)
ims = flopy.mf6.ModflowIms(
    gwf_sim,
    print_option="SUMMARY",
    complexity="complex",
    outer_dvclose=1.0e-8,
    inner_dvclose=1.0e-8,
)
disv = flopy.mf6.ModflowGwfdisv(
    gwf,
    nlay=mg.nlay,
    ncpl=mg.ncpl,
    nvert=mg.nvert,
    vertices=mg._vertices,
    cell2d=mg.cell2d,
    top=mg.top[0],
    botm=mg.top_botm[1][0],
)
gwf_ic = flopy.mf6.ModflowGwfic(gwf, strt=1.0)
gwf_sto = flopy.mf6.ModflowGwfsto(gwf, ss=0, sy=0, steady_state={0: True})
wells = [[0, c, -0.05, 80.0] for c in cells_wel]
wells[2][-2] *= -100  # convert third well to injection, Q = +5.0
gwf_wel = flopy.mf6.ModflowGwfwel(
    gwf,
    pname="WELL",
    auxiliary="TEMPERATURE",
    maxbound=len(wells),
    save_flows=True,
    stress_period_data={0: wells},
)
gwf_npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    xt3doptions=True,
    k=10.0,
    save_saturation=True,
    save_specific_discharge=True,
)

chdlist = []
seen = set()
for icpl in cells_left:
    yc = mg.cell2d[icpl][2]
    chdlist.append([(0, icpl), 2.0, 100.0 * yc / ymax])
    seen.add(int(icpl))
for icpl in cells_right:
    chdlist.append([(0, icpl), 1.0, 0.0])
    seen.add(int(icpl))
for icpl in cells_bottom:
    if int(icpl) not in seen:
        chdlist.append([(0, icpl), 1.8, 80.0])

gwf_chd = flopy.mf6.ModflowGwfchd(
    gwf,
    pname="CHD",
    auxiliary="TEMPERATURE",
    stress_period_data=chdlist,
)
gwf_oc = flopy.mf6.ModflowGwfoc(
    gwf,
    budget_filerecord=f"{gwf_name}.cbc",
    head_filerecord=f"{gwf_name}.hds",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
)

Write and run the flow simulation.

In [ ]:
gwf_sim.write_simulation(silent=False)
success, buff = gwf_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

Now construct a heat transport simulation. Again, we'll speed through this, both because this course hasn't covered GWE yet, and because PRT is our focus here.

But it's worth mentioning that the GWE model has a different time discretization than the flow model, running for 10⁶ days with 1000 time steps, with a slight multiplier, to approach steady state.

In [ ]:
import os

gwe_name = f"{example_name}-gwe"
gwe_ws = base_ws / "gwe"
gwe_ws.mkdir(exist_ok=True, parents=True)

gwe_sim = flopy.mf6.MFSimulation(sim_name=gwe_name, sim_ws=gwe_ws, exe_name="mf6")
gwe_tdis = flopy.mf6.ModflowTdis(
    gwe_sim, time_units=time_unit, perioddata=[[1.0e6, 1000, 1.003]]
)
gwe = flopy.mf6.MFModel(
    gwe_sim,
    model_type="gwe6",
    modelname=gwe_name,
    model_nam_file=f"{gwe_name}.name",
)
nouter = 1000
ninner = 200
hclose = 1e-6
rclose = 1e-6
relax = 1.0
imsgwe = flopy.mf6.ModflowIms(
    gwe_sim,
    print_option="SUMMARY",
    outer_dvclose=hclose,
    outer_maximum=nouter,
    under_relaxation="NONE",
    inner_maximum=ninner,
    inner_dvclose=hclose,
    rcloserecord=rclose,
    linear_acceleration="BICGSTAB",
    scaling_method="NONE",
    reordering_method="NONE",
    relaxation_factor=relax,
    filename=f"{gwe_name}.ims",
)
gwe_sim.register_ims_package(imsgwe, [gwe.name])
gwe_disv = flopy.mf6.ModflowGwedisv(
    gwe,
    nlay=mg.nlay,
    ncpl=mg.ncpl,
    nvert=mg.nvert,
    vertices=mg._vertices,
    cell2d=mg.cell2d,
    top=mg.top[0],
    botm=mg.top_botm[1][0],
)
strt_temp = 10.0  # initial temperature (°C)
alh = 0.0  # longitudinal mechanical dispersivity (m)
ath1 = 0.0  # transverse mechanical dispersivity (m)
ktw = 0.56 * 86400  # thermal conductivity of water, W/(m·°C) → J/(m·day·°C)
kts = 2.5 * 86400  # thermal conductivity of solids, W/(m·°C) → J/(m·day·°C)
rhow = 1000.0  # density of water (kg/m³)
cpw = 4180.0  # heat capacity of water (J/(kg·°C))
rhos = 2650.0  # density of dry solid (kg/m³)
cps = 900.0  # heat capacity of dry solid (J/(kg·°C))
lhv = 2500.0  # latent heat of vaporization (J/kg)
gwe_ic = flopy.mf6.ModflowGweic(gwe, strt=strt_temp)
gwe_adv = flopy.mf6.ModflowGweadv(gwe, scheme="TVD")
gwe_cnd = flopy.mf6.ModflowGwecnd(
    gwe,
    alh=alh,
    ath1=ath1,
    ktw=ktw,
    kts=kts,
)
porosity = 0.1
gwe_est = flopy.mf6.ModflowGweest(
    gwe,
    porosity=porosity,
    heat_capacity_water=cpw,
    density_water=rhow,
    latent_heat_vaporization=lhv,
    heat_capacity_solid=cps,
    density_solid=rhos,
)
gwe_ssm = flopy.mf6.ModflowGwessm(
    gwe,
    sources=[("WELL", "AUX", "TEMPERATURE"), ("CHD", "AUX", "TEMPERATURE")],
)
gwe_oc = flopy.mf6.ModflowGweoc(
    gwe,
    budget_filerecord=f"{gwe_name}.cbc",
    temperature_filerecord=f"{gwe_name}.ucn",
    saverecord={0: [("TEMPERATURE", "ALL"), ("BUDGET", "ALL")]},
    printrecord=[("TEMPERATURE", "LAST"), ("BUDGET", "LAST")],
)
rel_gwf_ws = os.path.relpath(gwf_ws, gwe_ws)
gwe_fmi = flopy.mf6.ModflowGwefmi(
    gwe,
    packagedata=[
        ("GWFHEAD", Path(f"{rel_gwf_ws}/{gwf_name}.hds"), None),
        ("GWFBUDGET", Path(f"{rel_gwf_ws}/{gwf_name}.cbc"), None),
    ],
)

Write and run the GWE simulation.

In [ ]:
gwe_sim.write_simulation(silent=False)
success, buff = gwe_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

Now to the good stuff, the particle tracking model.

In [ ]:
prt_name = f"{example_name}-prt"
prt_ws = base_ws / "prt"
prt_ws.mkdir(exist_ok=True, parents=True)

prt_sim = flopy.mf6.MFSimulation(
    sim_name=prt_name, version="mf6", exe_name="mf6", sim_ws=prt_ws
)
prt_tdis = flopy.mf6.ModflowTdis(
    prt_sim, time_units=time_unit, perioddata=[[1.0, 1, 1.0]]
)
prt = flopy.mf6.ModflowPrt(prt_sim, modelname=prt_name)
prt_disv = flopy.mf6.ModflowPrtdisv(
    prt,
    nlay=mg.nlay,
    ncpl=mg.ncpl,
    nvert=mg.nvert,
    vertices=mg._vertices,
    cell2d=mg.cell2d,
    top=mg.top[0],
    botm=mg.top_botm[1][0],
)
prt_mip = flopy.mf6.ModflowPrtmip(prt, pname="mip", porosity=porosity)

This model was adapted from a PRT test case that releases many evenly spaced particles along the left edge of the grid. We'll use just two of those release points. We use the FloPy `Grid.intersect()` method to determine the cell each point falls within. Unlike MP7, PRT requires release points to be specified in model rather than local coordinates. The release point must also fall within the specified cell. (Yes, it's redundant to provide both pieces of information. This may change in future.)

Note also the `extend_tracking` option. This instructs PRT to continue tracking particles beyond the simulation's end time, until the particles terminate naturally.

In [ ]:
rpts = [[20, i, 0.5] for i in range(1, 999, 20)]
prpdata = [
    (i, (0, mg.intersect(p[0], p[1])), p[0], p[1], p[2])
    for i, p in enumerate([rpts[23], rpts[32]])
]

prt_prp = flopy.mf6.ModflowPrtprp(
    prt,
    nreleasepts=len(prpdata),
    packagedata=prpdata,
    perioddata={0: ["FIRST"]},
    extend_tracking=True,
)

Continue with the remainder of the PRT model's packages. Becauuse this model postprocesses flow model results, we configure a flow model interface (FMI) package identifying the flow model's head and budget output files.

In [ ]:
prt_oc = flopy.mf6.ModflowPrtoc(
    prt,
    trackcsv_filerecord=[f"{prt_name}.trk.csv"],
)
prt_fmi = flopy.mf6.ModflowPrtfmi(
    prt,
    packagedata=[
        ("GWFHEAD", f"{rel_gwf_ws}/{gwf_name}.hds"),
        ("GWFBUDGET", f"{rel_gwf_ws}/{gwf_name}.cbc"),
    ],
)

Finally, configure an explicit model solution package. Unlike the other model types, which require an iterative solver, PRT "solves itself". If a PRT model is configured with an IMS instead of an EMS package, MF6 will raise an error.

In [ ]:
prt_ems = flopy.mf6.ModflowEms(prt_sim, pname="ems")
prt_sim.register_solution_package(prt_ems, [prt.name])

Write and run the particle tracking simulation.

In [ ]:
prt_sim.write_simulation(silent=False)
success, buff = prt_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

Load the temperature field and particle pathlines. Interpolate the GWE temperature at every particle position using a Clough–Tocher 2D interpolator built from the Voronoi cell centers.

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import CloughTocher2DInterpolator

temperatures = gwe.output.temperature().get_data()[0, 0]
pathlines = pd.read_csv(prt_ws / f"{prt_name}.trk.csv", na_filter=False)

xc = mg.xcellcenters
yc = mg.ycellcenters
interp = CloughTocher2DInterpolator(list(zip(xc, yc, strict=False)), temperatures)
pathlines["therm"] = interp(pathlines["x"].values, pathlines["y"].values)

particle_a = pathlines[pathlines["irpt"] == 1]
particle_b = pathlines[pathlines["irpt"] == 2]

Plot the results in map view, showing specific discharge vectors, the temperature field, and particle pathlines, as well as temperature vs. x coordinate and temperature vs. travel time.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

spdis = gwf.output.budget().get_data(text="DATA-SPDIS")[0]
qx, qy, _ = flopy.utils.postprocessing.get_specific_discharge(spdis, gwf)

fig, axes = plt.subplots(
    3,
    1,
    figsize=(6, 7),
    tight_layout=True,
    gridspec_kw={"height_ratios": [3, 1, 1]},
)

# map view
ax = axes[0]
ax.set_xlim(0, 2000)
ax.set_ylim(0, 1000)
ax.set_aspect("equal")
pmv = flopy.plot.PlotMapView(model=gwf, ax=ax)
pmv.plot_grid(alpha=0.25)
tempmesh = pmv.plot_array(temperatures, alpha=0.7)
cv = pmv.contour_array(temperatures, levels=np.linspace(0, 80, 9))
plt.clabel(cv, colors="k")
plt.colorbar(
    tempmesh,
    shrink=0.5,
    ax=ax,
    label="Temperature (°C)",
    location="bottom",
    fraction=0.1,
)
pmv.plot_vector(qx, qy, normalize=True, alpha=0.25)
pmv.plot_bc(ftype="WELL")
for ipl, (_, pl) in enumerate(pathlines.groupby(["iprp", "irpt", "trelease"])):
    pl.plot(
        kind="line", linestyle="--", x="x", y="y", ax=ax, legend=False, color="blue"
    )
    if ipl == 0:
        ax.annotate(
            "Particle 1",
            xy=(1050, 380),
            xycoords="data",
            xytext=(30, -20),
            textcoords="offset points",
            bbox={"boxstyle": "round", "fc": "1.0", "alpha": 0.66},
            arrowprops={
                "arrowstyle": "->",
                "shrinkA": 0,
                "shrinkB": 5,
                "connectionstyle": "angle,angleA=0,angleB=135,rad=40",
            },
        )
    else:
        ax.annotate(
            "Particle 2",
            xy=(1050, 610),
            xycoords="data",
            xytext=(-75, 10),
            textcoords="offset points",
            bbox={"boxstyle": "round", "fc": "1.0", "alpha": 0.66},
            arrowprops={
                "arrowstyle": "->",
                "shrinkA": 0,
                "shrinkB": 5,
                "connectionstyle": "angle,angleA=0,angleB=135,rad=30",
            },
        )
ax.legend(
    handles=[
        mpl.lines.Line2D(
            [0],
            [0],
            marker=">",
            linestyle="",
            color="grey",
            markerfacecolor="gray",
            label="Specific discharge",
        ),
        mpl.lines.Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markerfacecolor="red",
            label="Well",
        ),
    ],
    loc="lower right",
)

# shared colour norm for the two profile plots
norm = plt.Normalize(
    min(particle_a["therm"].min(), particle_b["therm"].min()) - 5,
    max(particle_a["therm"].max(), particle_b["therm"].max()) + 5,
)

# temperature vs x-position
for part in [particle_a, particle_b]:
    pts = np.array([part["x"], part["therm"]]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, norm=norm)
    lc.set_array(part["therm"])
    lc.set_linewidth(3)
    axes[1].add_collection(lc)
axes[1].annotate("Particle 2", xy=(400, 68), xycoords="data")
axes[1].annotate("Particle 1", xy=(400, 50), xycoords="data")
axes[1].set_xlabel("X (m)")
axes[1].set_xlim(0, 2000)
axes[1].set_xticks(np.arange(0, 2100, 250))
axes[1].set_ylabel("Temperature (°C)")
axes[1].set_ylim(40, 80)
axes[1].set_yticks(np.arange(40, 81, 10))

# temperature vs travel time
for part in [particle_a, particle_b]:
    pts = np.array([part["t"], part["therm"]]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, norm=norm)
    lc.set_array(part["therm"])
    lc.set_linewidth(3)
    axes[2].add_collection(lc)
axes[2].annotate("Particle 2", xy=(15000, 68), xycoords="data")
axes[2].annotate("Particle 1", xy=(15000, 50), xycoords="data")
axes[2].set_xlabel("Time (days)")
axes[2].set_xlim(0, max(particle_a["t"].max(), particle_b["t"].max()))
axes[2].set_ylabel("Temperature (°C)")
axes[2].set_ylim(40, 80)
axes[2].set_yticks(np.arange(40, 81, 10))

plt.show()